# Olist 이커머스 배송 지연 분석

**목표**: 배송 지연이 (1) 리뷰 점수, (2) 재구매율에 어떤 영향을 미치는지 통계적으로 검증

**데이터**: Brazilian E-Commerce Public Dataset by Olist (Kaggle) — SQLite(`olist.db`)에 적재 후 `orders_clean`, `reviews_clean` VIEW로 정제 (SQL 파일은 `/sql` 폴더 참고)

- `orders_clean`: 배송완료(delivered) + 날짜 이상치(2017-09-19, 282건) 제외된 주문 96,188건, 지연일수(`mi_date` = 실제배송일 - 예상배송일) 포함
- `reviews_clean`: 주문당 최신 리뷰 1건만 남긴 정제 리뷰 98,673건 (원본 대비 중복 551건 제거)


---
## 가설 1: 배송 지연이 리뷰 점수에 영향을 미치는가

**가설**: 배송이 지연될수록(mi_date > 0) 리뷰 점수(review_score, 1~5점)가 낮아질 것이다.

### 1-1. 데이터 로드
`orders_clean`과 `reviews_clean`을 `order_id` 기준 INNER JOIN하여 가져옴.   
(양쪽 다 리뷰/배송이 있는 주문만 대상 — 리뷰 없는 delivered 주문 642건은 자연스럽게 제외됨)


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/Users/gon-i/Desktop/olist.db')

query = """
select o.order_id, o.mi_date, r.review_score
from orders_clean o
join reviews_clean r on o.order_id = r.order_id
"""

df = pd.read_sql(query, conn)
conn.close()

df['delay_group'] = df['mi_date'].apply(lambda x: '지연' if x > 0 else '정상')

print(df.shape)
print(df['delay_group'].value_counts())

(95546, 4)
delay_group
정상    87937
지연     7609
Name: count, dtype: int64


In [2]:
df.head()

,order_id,mi_date,review_score,delay_group
0,e481f51cbdc54678b7cc49136f2d6af7,-7.107488,4,정상
1,53cdb2fc8bc7dce0b6741e2150273451,-5.355729,4,정상
2,47770eb9100c2d0c44946d9cf07ec65d,-17.245498,5,정상
3,949d5b44dbf5de918fe9c16f97b45f8a,-12.980069,5,정상
4,ad21c59c0840e6cb83a9ceb5573f8159,-9.238171,5,정상


In [3]:
df.describe()

,mi_date,review_score
count,95546.000000,95546.000000
mean,-11.258049,4.156249
std,9.867379,1.284609
min,-146.016123,1.000000
25%,-16.255764,4.000000
50%,-11.981181,5.000000
75%,-6.411042,5.000000
max,188.975081,5.000000


### 1-2. 교차표 및 카이제곱 검정

`delay_group`(지연/정상) × `review_score`(1~5점)의 교차표를 만들어 두 변수 간 관계를 검정.

In [13]:
from scipy import stats
import matplotlib.pyplot as plt

In [5]:
df_cross = pd.crosstab(df['delay_group'], df['review_score'])
df_cross

review_score,1,2,3,4,5
delay_group,,,,,
정상,5794,2310,7029,17897,54907
지연,3525,601,861,931,1691


In [6]:
chi2, p, dof, ef = stats.chi2_contingency(df_cross)

chi2, p, dof

(np.float64(14216.57172795135), np.float64(0.0), 4)

In [7]:
n = df_cross.values.sum()
r, c = df_cross.shape  

cramers_v = np.sqrt((chi2 / n) / min(r - 1, c - 1))
print(f"Cramér's V = {cramers_v:.4f}")

Cramér's V = 0.3857


### 1-3. 결과 해석

- **카이제곱 검정**: χ² = 14,216.57, p < 0.05 → 통계적으로 유의미
- **Cramér's V = 0.3857** → 중간~강한 수준의 관계 
- **방향**: 지연 그룹은 1점 비중(약 46%)이 압도적인 반면, 정상 그룹은 5점 비중(약 62%)이 압도적
  — 지연될수록 점수가 낮아지는 방향 확인

**결론**: 배송 지연은 리뷰 점수에 **강한 부정적 영향**을 미침.

---
## 가설 2: 배송 지연이 재구매율에 영향을 미치는가

**가설**: 배송 지연을 경험한 고객은 재구매율(customer_unique_id 기준 주문 2회 이상)이 낮을 것이다.

### 2-1. 첫 번째 시도 — 방향이 반대로 나온 문제

처음에는 고객의 **모든 주문 중 하나라도** 지연된 적이 있으면 `is_delayed = 1`로 정의했다.

```sql
with delay_repurchase as (
    select c.customer_unique_id
         , max(case when o.mi_date > 0 then 1 else 0 end) as is_delayed
         , count(distinct o.order_id) as order_cnt   
    from orders_clean o 
    left join customers c on o.customer_id = c.customer_id
    group by 1
), repurchase as (
    select is_delayed
         , sum(case when order_cnt >= 2 then 1 else 0 end) as repurchase
         , sum(case when order_cnt < 2 then 1 else 0 end) as n_repurhase
         , count(*) as cnt
    from delay_repurchase
    group by 1
)
select *
     , round(repurchase * 1.0 / cnt, 3) as ratio
from repurchase
```

**결과**: 정상 그룹 2.8% vs 지연 그룹 4.6% — 오히려 **지연을 겪은 고객의 재구매율이 더 높게** 나옴 -> 가설과 정반대 방향.

**의심**: `is_delayed`를 "모든 주문 중 하나라도 지연"으로 정의하면, **재구매를 이미 결정한 뒤에 일어난 지연**(예: 두 번째 주문의 지연)까지 원인 변수에 섞여 들어감.   
즉 재구매 여부가 결정되는 시점(첫 주문 직후)보다 **미래에 일어난 사건을 과거의 원인처럼** 취급한 셈 

**검증**: 지연 경험 유무별 평균 주문 횟수를 비교해봤지만 거의 차이가 없었음(1.03건 vs 1.05건) — 즉 "재구매를 많이 한 사람이라 지연을 겪을 기회도 많았다"는 설명으로는 역전 현상이 설명되지 않았고, 진짜 원인은 정의 자체의 시점 오류였음.


### 2-2. 재정의 — 첫 주문 기준으로 수정

재구매 여부에 영향을 줄 수 있는 지연은, 논리적으로 **재구매 결정 이전에 일어난 지연 = 첫 주문의 지연 여부** 임.   
`ROW_NUMBER()`로 고객별 첫 주문만 추출하여 `is_delayed`를 다시 정의함.

```sql
with first_purchase as (
    select customer_unique_id
         , mi_date
         , row_number() over(partition by customer_unique_id order by order_purchase_timestamp asc) as rn
         , count(*) over(partition by customer_unique_id) as cnt
    from orders_clean o 
    left join customers c on o.customer_id = c.customer_id
), first_purchase_flag as (
    select customer_unique_id
         , case when mi_date > 0 then 1 else 0 end as is_delayed
         , case when cnt >= 2 then 1 else 0 end as is_repurchase
    from first_purchase
    where rn = 1
)
select is_delayed
     , sum(case when is_repurchase = 1 then 1 else 0 end) as purchase_cnt
     , count(*) as total
from first_purchase_flag
group by is_delayed
```

**결과**: 정상 그룹 3.03%(2,594/85,538) vs 지연 그룹 2.49%(188/7,553) — 이번엔 가설과 **일치하는 방향**으로 나옴.

**검산**: 전체 모집단(93,091명)은 이전 시도와 동일하며, 165명이 그룹 재배정되었음.  
— "여러 주문 중 하나라도 지연"으로 봤을 때 지연 그룹에 잘못 포함됐던 165명이 "첫 주문은 정상"으로 재분류된 것임.

### 2-3. 카이제곱 검정 + Cramér's V

위 SQL 결과를 2×2 교차표로 옮겨 검정

In [19]:
df2_cross = pd.DataFrame([[82944, 2594], [7365, 188]], index = ['정상', '지연'] , columns = ['재구매 x', '재구매 o'])
df2_cross

,재구매 x,재구매 o
정상,82944,2594
지연,7365,188


In [20]:
chi2, p, dof, ef = stats.chi2_contingency(df2_cross)

chi2, p, dof

(np.float64(6.8848648833951165), np.float64(0.008692863875619341), 1)

In [21]:
import numpy as np

n = df2_cross.values.sum()
r, c = df2_cross.shape  

cramers_v = np.sqrt((chi2 / n) / min(r - 1, c - 1))
print(f"Cramér's V = {cramers_v:.4f}")

Cramér's V = 0.0086


### 2-4. 결과 해석

- **카이제곱 검정**: χ² = 6.88, p = 0.0087 → 0.05 기준 통계적으로 유의미
- **Cramér's V = 0.0086** → 거의 무관 수준 (0.1 미만)

**표본 쏠림 때문인가, 실제 차이가 작아서인가?**
- 정상(85,538) : 지연(7,553) ≈ 11:1로 어느 정도 불균형은 있지만, 지연 그룹도 7,553명으로 표본 자체는 충분히 큼
   — 극단적 쏠림(예: 1% 미만)은 아님.
- 재구매율 차이(3.03% vs 2.49%, 0.54%p)는 표본이 커서(93,091명) 통계적으로는 유의하게 잡히지만, **실제 행동 차이 자체가 원래 작기 때문**이지 표본 쏠림 때문이 아님.

**결론**: 배송 지연은 재구매율에 방향상 부정적 영향을 미치지만(가설과 일치), **효과크기가 매우 작아 실질적인 영향력은 미미**함.